<a href="https://colab.research.google.com/github/fourmodern/2025_aidrugdiscovery/blob/main/Day5/t110_esm2_peptide_optimization_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ESM-2 Protein-Peptide Binding Optimization Tutorial

> **2026 갱신**: Colab(Python 3.12, torch 2.11.0+cu128 사전설치) 기준으로 최신화. torch는 재설치하지 않고 `transformers`만 설치하며, 결합 최적화 라이브러리 `evo_prot_grad`(최신 0.2)는 `--no-deps`로 설치해 Colab의 최신 transformers를 유지합니다(0.2가 `transformers==4.38.0`을 고정하는 다운그레이드 방지). 모델 id(`ChatterjeeLab/PepMLM-650M`, `facebook/esm2_t6_8M_UR50D`)와 evo_prot_grad API(`get_expert`, `DirectedEvolution`, `scoring_strategy='mutant_marginal'`)는 공식 소스로 실재 확인. numpy 2.x 호환.

This notebook demonstrates how to use ESM-2, a protein language model from Facebook AI Research, to generate and optimize peptide binders for target proteins. The workflow includes:
1. Setting up the environment
2. Loading the ESM-2 model
3. Preparing a protein sequence
4. Generating peptide sequences
5. Optimizing binding affinity with evolutionary strategies

---


In [ ]:
# Step 1: Setup
# 필요한 라이브러리 설치
#
# 주의 1) PyPI의 "esm" 패키지는 EvolutionaryScale의 ESM3(ESM-2가 아님)입니다.
#         이 노트북은 transformers로 ESM-2 계열 모델을 불러오므로 esm 패키지가 필요 없습니다.
# 주의 2) 2026 현행 Colab에는 torch 2.11.0+cu128 이 사전설치되어 있으므로 torch를 재설치하지 않습니다.
#         (torch 재설치는 CUDA 빌드 충돌을 유발할 수 있습니다.)
# 최신 transformers를 그대로 사용합니다(ESM/EsmForMaskedLM API는 안정적).
# 재현성이 중요하면 아래처럼 하한을 고정할 수 있습니다:
#   !pip install -q "transformers>=4.44"
!pip install -q transformers


### Step 1: Environment Setup
Hugging Face 모델을 다루기 위한 `transformers`만 설치합니다. `torch`는 현행 Colab(2.11.0+cu128)에 이미 설치되어 있어 재설치하지 않습니다. 단백질 언어모델은 transformers의 ESM 구현으로 불러오므로 PyPI의 `esm`(ESM3) 패키지는 필요하지 않습니다.


In [ ]:

# Step 2: 펩타이드 생성 모델(ESM-2 기반) 로드
# ChatterjeeLab/PepMLM-650M 은 ESM-2(650M) 를 파인튜닝한 binder 생성용 MLM 입니다.
# (원 저장소가 TianlaiChen -> ChatterjeeLab 으로 이전되었습니다. HF 실재/공개 확인됨.)
# 최초 실행 시 약 2.5GB 다운로드가 발생하며, GPU 런타임 권장.
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import pandas as pd
import numpy as np
from torch.distributions import Categorical

# Hugging Face Hub에서 사전학습 모델과 토크나이저 로드
model = AutoModelForMaskedLM.from_pretrained("ChatterjeeLab/PepMLM-650M")
tokenizer = AutoTokenizer.from_pretrained("ChatterjeeLab/PepMLM-650M")

print("Model and tokenizer loaded successfully.")



### Step 2: Load ESM-2 기반 모델
Hugging Face Hub의 사전학습 모델 `ChatterjeeLab/PepMLM-650M` 을 사용합니다. 이 모델은 ESM-2(650M) 단백질 언어모델을 파인튜닝한 것으로, 표적 단백질 서열에 결합하는 펩타이드(binder) 영역을 마스크드 언어모델링(MLM)으로 재구성하도록 학습되었습니다.


In [ ]:

# Step 3: Prepare a protein sequence
# Example protein sequence from UniProt (replace with an actual sequence)
protein_seq = "MSGIALSRLAQERKAWRKDHPFGFVAVPTKNPDGTMNLMNWECAIPGKKGTPWEGGLFKLRMLFKDDYPSSPPKCKFEPPLFHPNVYPSGTVCLSILEEDKDWRPAITIKQILLGIQELLNEPNIQDPAQAEAYTIYCQNRVEYEKRVRAQAKKFAPS"

# Tokenize the protein sequence for the model
inputs = tokenizer(protein_seq, return_tensors="pt")

print("Protein sequence tokenized.")
print("Inputs:", inputs)



### Step 3: Generate Peptide Sequence
Using PepMLM (Masked Language Modeling), we can generate a peptide sequence that is predicted to bind the input protein. The model predicts the most suitable peptide based on the given protein sequence.


In [ ]:

def compute_pseudo_perplexity(model, tokenizer, protein_seq, binder_seq):
    sequence = protein_seq + binder_seq
    original_input = tokenizer.encode(sequence, return_tensors='pt').to(model.device)
    length_of_binder = len(binder_seq)

    # Prepare a batch with each row having one masked token from the binder sequence
    masked_inputs = original_input.repeat(length_of_binder, 1)
    positions_to_mask = torch.arange(-length_of_binder - 1, -1, device=model.device)

    masked_inputs[torch.arange(length_of_binder), positions_to_mask] = tokenizer.mask_token_id

    # Prepare labels for the masked tokens
    labels = torch.full_like(masked_inputs, -100)
    labels[torch.arange(length_of_binder), positions_to_mask] = original_input[0, positions_to_mask]

    # Get model predictions and calculate loss
    with torch.no_grad():
        outputs = model(masked_inputs, labels=labels)
        loss = outputs.loss

    # Loss is already averaged by the model
    avg_loss = loss.item()
    pseudo_perplexity = np.exp(avg_loss)
    return pseudo_perplexity


def generate_peptide_for_single_sequence(protein_seq, peptide_length = 15, top_k = 3, num_binders = 4):

    peptide_length = int(peptide_length)
    top_k = int(top_k)
    num_binders = int(num_binders)

    binders_with_ppl = []

    for _ in range(num_binders):
        # Generate binder
        masked_peptide = '<mask>' * peptide_length
        input_sequence = protein_seq + masked_peptide
        inputs = tokenizer(input_sequence, return_tensors="pt").to(model.device)

        with torch.no_grad():
            logits = model(**inputs).logits
        mask_token_indices = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
        logits_at_masks = logits[0, mask_token_indices]

        # Apply top-k sampling
        top_k_logits, top_k_indices = logits_at_masks.topk(top_k, dim=-1)
        probabilities = torch.nn.functional.softmax(top_k_logits, dim=-1)
        predicted_indices = Categorical(probabilities).sample()
        predicted_token_ids = top_k_indices.gather(-1, predicted_indices.unsqueeze(-1)).squeeze(-1)

        generated_binder = tokenizer.decode(predicted_token_ids, skip_special_tokens=True).replace(' ', '')

        # Compute PPL for the generated binder
        ppl_value = compute_pseudo_perplexity(model, tokenizer, protein_seq, generated_binder)

        # Add the generated binder and its PPL to the results list
        binders_with_ppl.append([generated_binder, ppl_value])

    return binders_with_ppl

def generate_peptide(input_seqs, peptide_length=15, top_k=3, num_binders=4):
    if isinstance(input_seqs, str):  # Single sequence
        binders = generate_peptide_for_single_sequence(input_seqs, peptide_length, top_k, num_binders)
        return pd.DataFrame(binders, columns=['Binder', 'Pseudo Perplexity'])

    elif isinstance(input_seqs, list):  # List of sequences
        results = []
        for seq in input_seqs:
            binders = generate_peptide_for_single_sequence(seq, peptide_length, top_k, num_binders)
            for binder, ppl in binders:
                results.append([seq, binder, ppl])
        return pd.DataFrame(results, columns=['Input Sequence', 'Binder', 'Pseudo Perplexity'])


### Step 5: Optimize Peptide Binding Affinity
We employ an evolutionary strategy, such as EvoProtGrad, to refine the generated peptide sequence. The goal is to enhance the binding affinity between the peptide and the target protein.


In [ ]:
results_df = generate_peptide(protein_seq, peptide_length=15, top_k=3, num_binders=5)
print(results_df)

## In Silico Directed Evolution of the Peptide Binder with EvoProtGrad and ESM-2

In [ ]:
# evo_prot_grad 설치 (최신 0.2 — 결합 최적화용 in silico directed evolution)
#
# 두 가지 주의사항:
# 1) 셸에서 !pip install evo_prot_grad>=0.2 처럼 쓰면 ">"가 리다이렉션으로 해석되어
#    버전 조건이 무시되고 "=0.2" 파일이 생깁니다. 반드시 따옴표로 묶습니다.
# 2) evo_prot_grad 는 transformers[torch]==4.38.0 을 정확히 고정합니다. 그대로 설치하면
#    Colab의 최신 transformers 가 2024년 4.38.0 으로 강제 다운그레이드되어 앞 단계가 깨집니다.
#    --no-deps 로 설치해 현재 transformers/torch 를 유지합니다.
#    (evo_prot_grad 의 유일한 추가 런타임 의존성은 pandas 이며 Colab 에 사전설치되어 있습니다. numpy 2.x 호환.)
!pip install -q --no-deps "evo_prot_grad>=0.2"

#del model
#torch.cuda.empty_cache()
import torch
import evo_prot_grad
from transformers import AutoTokenizer, EsmForMaskedLM

In [ ]:
def run_evo_prot_grad_on_paired_sequence(paired_protein_sequence):
    # Replace ':' with a string of 20 'G' amino acids
    separator = 'G' * 20
    sequence_with_separator = paired_protein_sequence.replace(':', separator)

    # Determine the start and end indices of the first protein and the separator
    separator_start_index = sequence_with_separator.find(separator)
    first_protein_end_index = separator_start_index
    separator_end_index = separator_start_index + len(separator)

    # Format the sequence into FASTA format
    fasta_format_sequence = f">Paired_Protein_Sequence\n{sequence_with_separator}"

    # Save the sequence to a temporary file
    temp_fasta_path = "temp_paired_sequence.fasta"
    with open(temp_fasta_path, "w") as file:
        file.write(fasta_format_sequence)

    # 메모리 절약을 위해 가장 작은 ESM-2 (t6_8M) 를 expert 로 사용합니다.
    # 더 높은 정확도가 필요하면 아래 대안으로 교체 가능(HF 실재 확인):
    #   facebook/esm2_t12_35M_UR50D  (35M)
    #   facebook/esm2_t33_650M_UR50D (650M, GPU 권장)
    esm2_expert = evo_prot_grad.get_expert(
        'esm',
        model=EsmForMaskedLM.from_pretrained("facebook/esm2_t6_8M_UR50D"),  # 작은 ESM-2 (8M)
        tokenizer=AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D"),
        # pseudolikelihood_ratio는 최신 transformers에서 텐서 차원 불일치로 실패합니다.
        # mutant_marginal은 EvoProtGrad 논문에서 권장하는 전략이며 정상 동작합니다.
        scoring_strategy='mutant_marginal',
        temperature=0.95,
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )

    # Initialize wildtype sequence for the expert with the correct format
    wildtype_sequence = sequence_with_separator.replace(" ", "")  # Make sure the input sequence has no spaces
    esm2_expert.init_wildtype(wildtype_sequence)

    # Initialize Directed Evolution with the preserved first protein and separator region
    directed_evolution = evo_prot_grad.DirectedEvolution(
        wt_fasta=temp_fasta_path,
        output='best',
        experts=[esm2_expert],
        parallel_chains=1,  # Reduce parallel chains to save memory
        n_steps=50,
        max_mutations=15,
        verbose=True,
        preserved_regions=[(0, first_protein_end_index), (separator_start_index, separator_end_index)]
    )

    # Run the evolution process
    variants, scores = directed_evolution()

    # Process the results and split them into Protein 1 and Protein 2
    for variant, score in zip(variants, scores):
        # Remove spaces from the sequence
        evolved_sequence_no_spaces = variant.replace(" ", "")

        # Split the sequence at the separator
        protein_1, protein_2 = evolved_sequence_no_spaces.split(separator)

        print(f"Protein: {protein_1}, Evolved Peptide: {protein_2}, Score: {score}")

In [ ]:
# Example usage
paired_protein_sequence = "MSGIALSRLAQERKAWRKDHPFGFVAVPTKNPDGTMNLMNWECAIPGKKGTPWEGGLFKLRMLFKDDYPSSPPKCKFEPPLFHPNVYPSGTVCLSILEEDKDWRPAITIKQILLGIQELLNEPNIQDPAQAEAYTIYCQNRVEYEKRVRAQAKKFAPS:FDEDDPLAPRLLEEE"  # Replace with your paired protein sequences
run_evo_prot_grad_on_paired_sequence(paired_protein_sequence)